In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pickle

In [2]:
# Load a sample of the Avazu data (first 1 million rows)
# The full file is 40 million rows - too large for quick training


df = pd.read_csv('/Users/mohau/Desktop/STADIOalot-Capstone/datasets/train.csv', nrows=1000000)
# Display basic information
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.info())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df['click'].value_counts())

Shape: (1000000, 24)

First 5 rows:
             id  click      hour    C1  banner_pos   site_id site_domain  \
0  1.000009e+18      0  14102100  1005           0  1fbe01fe    f3845767   
1  1.000017e+19      0  14102100  1005           0  1fbe01fe    f3845767   
2  1.000037e+19      0  14102100  1005           0  1fbe01fe    f3845767   
3  1.000064e+19      0  14102100  1005           0  1fbe01fe    f3845767   
4  1.000068e+19      0  14102100  1005           1  fe8cc448    9166c161   

  site_category    app_id app_domain  ... device_type device_conn_type    C14  \
0      28905ebd  ecad2386   7801e8d9  ...           1                2  15706   
1      28905ebd  ecad2386   7801e8d9  ...           1                0  15704   
2      28905ebd  ecad2386   7801e8d9  ...           1                0  15704   
3      28905ebd  ecad2386   7801e8d9  ...           1                0  15706   
4      0569f928  ecad2386   7801e8d9  ...           1                0  18993   

   C15  C16   C17  C

In [3]:
# Extract time features from the raw 'hour' column BEFORE scaling
df['datetime'] = pd.to_datetime(df['hour'].astype(str), format='%y%m%d%H')
df['day_of_week'] = df['datetime'].dt.dayofweek
df['hour_of_day'] = df['datetime'].dt.hour
df = df.drop('datetime', axis=1)

print("Time features added")
print(df[['hour', 'day_of_week', 'hour_of_day']].head())
print("Shape after time features:", df.shape)

Time features added
       hour  day_of_week  hour_of_day
0  14102100            1            0
1  14102100            1            0
2  14102100            1            0
3  14102100            1            0
4  14102100            1            0
Shape after time features: (1000000, 26)


In [4]:
# Drop the id column (no predictive value)
df = df.drop('id', axis=1)

# Check target distribution
print("Target distribution:")
print(df['click'].value_counts())
print("\nTarget proportion:")
print(df['click'].value_counts(normalize=True))

Target distribution:
click
0    839781
1    160219
Name: count, dtype: int64

Target proportion:
click
0    0.839781
1    0.160219
Name: proportion, dtype: float64


In [5]:
# Separate features and target
X = df.drop('click', axis=1)
y = df['click']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000000, 24)
y shape: (1000000,)


In [6]:
# Identify categorical columns (object type)
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)

# Apply Label Encoding to each categorical column
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print("\nAfter encoding:")
print(X.dtypes)

Categorical columns: ['site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip', 'device_model']

After encoding:
hour                int64
C1                  int64
banner_pos          int64
site_id             int64
site_domain         int64
site_category       int64
app_id              int64
app_domain          int64
app_category        int64
device_id           int64
device_ip           int64
device_model        int64
device_type         int64
device_conn_type    int64
C14                 int64
C15                 int64
C16                 int64
C17                 int64
C18                 int64
C19                 int64
C20                 int64
C21                 int64
day_of_week         int32
hour_of_day         int32
dtype: object


In [7]:
# Split into 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (800000, 24)
X_test shape: (200000, 24)
y_train shape: (800000,)
y_test shape: (200000,)


In [ ]:
# Scale features using MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print("X_train_scaled shape:", X_train_scaled.shape)

In [ ]:
base_path = '/Users/mohau/Desktop/STADIOalot-Capstone/datasets/'

# Save the processed data
with open(base_path + 'X_train.pkl', 'wb') as f:
    pickle.dump(X_train_scaled, f)

with open(base_path + 'X_test.pkl', 'wb') as f:
    pickle.dump(X_test_scaled, f)

with open(base_path + 'y_train.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open(base_path + 'y_test.pkl', 'wb') as f:
    pickle.dump(y_test, f)

print("Processed data saved successfully!")
print("Files saved to:", base_path)